In [90]:

import pandas as pd
import numpy as np
import os
import re
import datetime

# Data manipulation and analysis
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning libraries
from sklearn.preprocessing import StandardScaler, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

# Additional utilities
import warnings
warnings.filterwarnings('ignore')

# Set style for visualizations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.impute import KNNImputer
# Set visualization style
sns.set_style("darkgrid")
sns.set_palette("husl")


In [91]:

def clean_climate_data(file_path, output_column_name):
    """
    Cleans climate data Excel files and returns a standardized DataFrame.
    
    Parameters:
    - file_path: Path to Excel file
    - output_column_name: Name for the measurement column (e.g., 'Rainfall')
    
    Returns:
    - DataFrame with District, Station, Year, Month, Week, and measurement columns
    """
    # Read Excel file without assuming headers
    raw_df = pd.read_excel(file_path, header=None)
    
    # Initialize district and station variables
    district = station = None
    
    # Search first 5 rows and columns for district/station info
    for i in range(5):
        for j in range(5):
            cell_value = str(raw_df.iloc[i, j])
            if 'District' in cell_value:
                district = cell_value.split(':')[1].strip() if ':' in cell_value else 'Unknown'
            if 'Station' in cell_value:
                station = cell_value.split(':')[-1].strip() if ':' in cell_value else 'Unknown'
    
    print(district)
    # Fallback if district not found
    district = district if district else station
    # Process yearly data blocks
    cleaned_rows = []
    current_row = 0
    
    while current_row < len(raw_df):
        row_data = raw_df.iloc[current_row]
        first_cell = str(row_data.iloc[0]).strip()
        
        # Check if row starts a new year block (4-digit year)
        if re.match(r'^\d{4}$', first_cell):
            year = int(first_cell)
            months = raw_df.iloc[current_row].iloc[1:13].tolist()
            weekly_data = raw_df.iloc[current_row+1:current_row+5].reset_index(drop=True)
            
            # Process each week (4 weeks per month)
            for week_number in range(4):
                week_label = weekly_data.iloc[week_number, 0]
                for month_idx in range(12):
                    month_name = months[month_idx]
                    measurement = weekly_data.iloc[week_number, month_idx + 1]
                    
                    cleaned_rows.append({
                        'District': district,
                        'Station': station,
                        'Year': year,
                        'Month': month_name,
                        'Week': week_label,
                        output_column_name: measurement
                    })
            
            current_row += 5  # Skip to next year block
        else:
            current_row += 1  # Move to next row
    
    return pd.DataFrame(cleaned_rows)


In [112]:

# Configuration
RAINFALL_DIR = '../RawDataset/rainfall'
rainfall_data_frames = []

# Process each rainfall file
print("Processing rainfall data files:")
for filename in os.listdir(RAINFALL_DIR):
    if filename.endswith('.xlsx'):
        file_path = os.path.join(RAINFALL_DIR, filename)
        print(f"- {filename}")
        df = clean_climate_data(file_path, "Rainfall")
        rainfall_data_frames.append(df)

# Combine all rainfall data
combined_rainfall_df = pd.concat(rainfall_data_frames, ignore_index=True)

# Save combined data
rainfall_output_path = '../RawDataset/rainfall/combined_rainfall_data.csv'
combined_rainfall_df.to_csv(rainfall_output_path, index=False)
print(f"\nSaved combined rainfall data to {rainfall_output_path}")

# Show sample data
print("\nSample rainfall data:")
combined_rainfall_df.tail()

Processing rainfall data files:
- rain_unguja_magharibi_b.xlsx
MAGHARIBI  B
- WETE.xlsx
WETE
- rain_unguja_kusini.xlsx
KUSINI
- mjini.xlsx
MJINI
- Micheweni.xlsx
MICHEWENI
- rain_unguja_kas_b.xlsx
KASKAZINI B
- rain_unguja_magharibi_a.xlsx
MAGHARIBI  A
- rain_unguja_kas_a.xlsx
KASKAZINI A
- Mkoani.xlsx
MKOANI
- Chake-Chake.xlsx
CHAKE-CHAKE

Saved combined rainfall data to ../RawDataset/rainfall/combined_rainfall_data.csv

Sample rainfall data:


,District,Station,Year,Month,Week,Rainfall
2875,CHAKE-CHAKE,Karume airport Pemba,2018,Aug,4th week,0.0
2876,CHAKE-CHAKE,Karume airport Pemba,2018,Sep,4th week,38.5
2877,CHAKE-CHAKE,Karume airport Pemba,2018,Oct,4th week,61.3
2878,CHAKE-CHAKE,Karume airport Pemba,2018,Nov,4th week,3.3
2879,CHAKE-CHAKE,Karume airport Pemba,2018,Dec,4th week,0.6


In [145]:

# Process maximum temperature data
MAX_TEMP_DIR = '../RawDataset/temperatures/max_temp'
max_temp_data_frames = []

print("\nProcessing maximum temperature data files:")
for filename in os.listdir(MAX_TEMP_DIR):
    if filename.endswith('.xlsx'):
        file_path = os.path.join(MAX_TEMP_DIR, filename)
        print(f"- {filename}")
        df = clean_climate_data(file_path, "Max Temperature")
        max_temp_data_frames.append(df)

combined_max_temp_df = pd.concat(max_temp_data_frames, ignore_index=True)
max_temp_output_path = '../RawDataset/temperatures/max_temp/combined_max_temperature_data.csv'
combined_max_temp_df.to_csv(max_temp_output_path, index=False)
print(f"\nSaved combined max temperature data to {max_temp_output_path}")

# Process minimum temperature data
MIN_TEMP_DIR = '../RawDataset/temperatures/min_temp'
min_temp_data_frames = []

print("\nProcessing minimum temperature data files:")
for filename in os.listdir(MIN_TEMP_DIR):
    if filename.endswith('.xlsx'):
        file_path = os.path.join(MIN_TEMP_DIR, filename)
        print(f"- {filename}")
        df = clean_climate_data(file_path, "Min Temperature")
        min_temp_data_frames.append(df)

combined_min_temp_df = pd.concat(min_temp_data_frames, ignore_index=True)
min_temp_output_path = '../RawDataset/temperatures/min_temp/combined_min_temperature_data.csv'
combined_min_temp_df.to_csv(min_temp_output_path, index=False)
print(f"\nSaved combined min temperature data to {min_temp_output_path}")

# Show sample data
print("\nSample temperature data:")
print("Max Temperature:")
print(combined_min_temp_df.tail())
print(combined_max_temp_df.tail())

#Define the columns to match for merging
merge_keys = ['District', 'Station', 'Year', 'Month', 'Week']

# Merge the dataframes
merged_temperature_df = pd.merge(
    combined_min_temp_df,
    combined_max_temp_df[merge_keys + ['Max Temperature']],
    on=merge_keys,
    how='left'  # or 'inner' if you want only matching rows
)

# Save the final merged dataset
final_output_path = '../RawDataset/temperatures/combined_temperature_data.csv'
merged_temperature_df.to_csv(final_output_path, index=False)

# Print sample
print("\nSample of final merged temperature data:")
print(merged_temperature_df)



Processing maximum temperature data files:
- unguja_max_temp.xlsx
None
- Maximum. t pemba.xlsx
None

Saved combined max temperature data to ../RawDataset/temperatures/max_temp/combined_max_temperature_data.csv

Processing minimum temperature data files:
- unguja_min_temp.xlsx
None
- MINIMUM pemba.xlsx
None

Saved combined min temperature data to ../RawDataset/temperatures/min_temp/combined_min_temperature_data.csv

Sample temperature data:
Max Temperature:
    District Station  Year Month      Week  Min Temperature
523    Pemba   Pemba  2018   Aug  4th week        24.671429
524    Pemba   Pemba  2018   Sep  4th week        23.200000
525    Pemba   Pemba  2018   Oct  4th week        23.757143
526    Pemba   Pemba  2018   Nov  4th week        24.671429
527    Pemba   Pemba  2018   Dec  4th week        24.842857
    District Station  Year Month      Week  Max Temperature
523    Pemba   Pemba  2018   Aug  4th week        28.657143
524    Pemba   Pemba  2018   Sep  4th week        29.37142

In [146]:
import pandas as pd

# Example dataframe:
# df = pd.read_csv("your_file.csv")

# Month and Week mappings
month_to_number = {
    'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4,
    'May': 5, 'Jun': 6, 'Jul': 7, 'Aug': 8,
    'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12
}

week_to_number = {
    '1st week': 0,
    '2nd week': 1,
    '3rd week': 2,
    '4th week': 3,
    '5th week': 4  # Handle months with 5 weeks
}
df = merged_temperature_df
# Apply mappings safely
df['MonthNum'] = df['Month'].map(month_to_number)
df['WeekNum'] = df['Week'].map(week_to_number)

# Drop rows with missing mappings (in case of invalid labels)
df = df.dropna(subset=['MonthNum', 'WeekNum'])

# Compute WeekOfYear
df['Week'] = (df['MonthNum'] - 1) * 4 + df['WeekNum'] + 1

# Handle possible overflow beyond 52 weeks
df['Week'] = df['Week'].clip(upper=53)
merged_temperature_df=df
merged_temperature_df

     District   Station  Year Month  Week  Min Temperature  Max Temperature  \
0    Zanzibar  Zanzibar  2023   Jan     1        26.885714        32.142857   
1    Zanzibar  Zanzibar  2023   Feb     5        25.042857        33.357143   
2    Zanzibar  Zanzibar  2023   Mar     9        24.771429        33.914286   
3    Zanzibar  Zanzibar  2023   Apr    13        25.528571        31.585714   
4    Zanzibar  Zanzibar  2023   May    17        24.657143        30.885714   
..        ...       ...   ...   ...   ...              ...              ...   
523     Pemba     Pemba  2018   Aug    32        24.671429        28.657143   
524     Pemba     Pemba  2018   Sep    36        23.200000        29.371429   
525     Pemba     Pemba  2018   Oct    40        23.757143        29.671429   
526     Pemba     Pemba  2018   Nov    44        24.671429        31.485714   
527     Pemba     Pemba  2018   Dec    48        24.842857        30.771429   

     MonthNum  WeekNum  
0           1        0  
1

In [122]:
# Standardize temperature data frames
# combined_max_temp_df["Region"] = combined_max_temp_df["Station"]
combined_max_temp_df.drop(["Station", "District"], axis=1, inplace=True)

# combined_min_temp_df["Region"] = combined_min_temp_df["Station"]
combined_min_temp_df.drop(["Station", "District"], axis=1, inplace=True)

# Define region mapping
PEMBA_DISTRICTS = ['WETE', 'MICHEWENI', 'CHAKE-CHAKE', 'MKOANI']
UNGUJA_DISTRICTS = ['MAGHARIBI  B', 'KUSINI', 'MJINI', 'KASKAZINI B', 'MAGHARIBI  A', 'KASKAZINI A', 'KATI']

def map_to_island(district):
    """Maps district name to broader region (Zanzibar/Pemba)."""
    if pd.isna(district):
        return 'Unknown'
    district_upper = district.upper()
    if district_upper in [d.upper() for d in UNGUJA_DISTRICTS]:
        return 'Unguja'
    elif district_upper in [d.upper() for d in PEMBA_DISTRICTS]:
        return 'Pemba'
    return 'Unknown'

# Apply region mapping to rainfall data
combined_rainfall_df['Island'] = combined_rainfall_df['District'].apply(map_to_island)

# Merge temperature data
temperature_combined_df = pd.merge(
    combined_max_temp_df,
    combined_min_temp_df,
    on=['Year', 'Month', 'Week'],
    how='outer'
)

# Final merge with rainfall data
climate_combined_df = pd.merge(
    combined_rainfall_df,
    temperature_combined_df,
    on=['Year', 'Month', 'Week'],
    how='left'
)

# Save final combined data
final_output_path = '../RawDataset/rainfall_temperature.csv'
climate_combined_df.to_csv(final_output_path, index=False)
print(f"\nSaved final combined climate data to {final_output_path}")

# Show final data structure
print("\nFinal combined data structure:")
climate_combined_df.info()
climate_combined_df.head()

# Define the columns to match for merging
merge_keys = ['District', 'Station', 'Year', 'Month', 'Week']

# Merge the dataframes
merged_temperature_df = pd.merge(
    combined_min_temp_df,
    combined_max_temp_df[merge_keys + ['Max Temperature']],
    on=merge_keys,
    how='left'  # or 'inner' if you want only matching rows
)

# Save the final merged dataset
final_output_path = '../RawDataset/temperatures/combined_temperature_data.csv'
merged_temperature_df.to_csv(final_output_path, index=False)

# Print sample
print("\nSample of final merged temperature data:")
print(merged_temperature_df.tail())



Saved final combined climate data to ../RawDataset/rainfall_temperature.csv

Final combined data structure:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10080 entries, 0 to 10079
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   District         10080 non-null  object 
 1   Station          10080 non-null  object 
 2   Year             10080 non-null  int64  
 3   Month            10080 non-null  object 
 4   Week             10080 non-null  object 
 5   Rainfall         9840 non-null   float64
 6   Island           10080 non-null  object 
 7   Max Temperature  10080 non-null  float64
 8   Min Temperature  10080 non-null  float64
dtypes: float64(3), int64(1), object(5)
memory usage: 708.9+ KB


KeyError: "['District', 'Station'] not in index"

In [123]:
# Week mapping helper
df = climate_combined_df
week_order = {'1st week': 1, '2nd week': 2, '3rd week': 3, '4th week': 4, '5th week': 5}
month_order = {'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4, 'May': 5, 'Jun': 6,
               'Jul': 7, 'Aug': 8, 'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12}

# Map the text to integers
df['Month_num'] = df['Month'].map(month_order)
df['Week_num'] = df['Week'].map(week_order)

# Calculate week of the year
df['WeekOfYear'] = (df['Month_num'] - 1) * 4 + df['Week_num']

# Drop helper columns if you don't need them
df.drop(columns=['Month_num', 'Week_num'], inplace=True)

print(df[['Year', 'Month', 'Week', 'WeekOfYear']])

       Year Month      Week  WeekOfYear
0      2023   Jan  1st week           1
1      2023   Jan  1st week           1
2      2023   Jan  1st week           1
3      2023   Jan  1st week           1
4      2023   Feb  1st week           5
...     ...   ...       ...         ...
10075  2018   Aug  4th week          32
10076  2018   Sep  4th week          36
10077  2018   Oct  4th week          40
10078  2018   Nov  4th week          44
10079  2018   Dec  4th week          48

[10080 rows x 4 columns]


In [95]:

def predict_temperature(group_df, temperature_column):
    """
    Predicts temperature for 2018 using linear regression on historical data.
    
    Returns:
    - Predicted temperature or None if insufficient data
    """
    clean_df = group_df.dropna(subset=[temperature_column])
    if clean_df['Year'].nunique() < 2:
        return None  # Not enough data points
    
    model = LinearRegression()
    X = clean_df['Year'].values.reshape(-1, 1)
    y = clean_df[temperature_column].values
    model.fit(X, y)
    return model.predict([[2018]])[0]

# Process each district-month-week combination
for (district, month, week), group in climate_combined_df.groupby(['District', 'Month', 'Week']):
    # Find 2018 records for this group
    mask_2018 = (
        (climate_combined_df['Year'] == 2018) &
        (climate_combined_df['District'] == district) &
        (climate_combined_df['Month'] == month) &
        (climate_combined_df['Week'] == week)
    )
    
    if mask_2018.any():
        # Get historical data (excluding 2018)
        historical_data = climate_combined_df[
            (climate_combined_df['District'] == district) &
            (climate_combined_df['Month'] == month) &
            (climate_combined_df['Week'] == week) &
            (climate_combined_df['Year'] != 2018)
        ]
        
        # Predict missing values
        predicted_max = predict_temperature(historical_data, 'Max Temperature')
        predicted_min = predict_temperature(historical_data, 'Min Temperature')
        
        # Update DataFrame
        if predicted_max is not None:
            climate_combined_df.loc[mask_2018, 'Max Temperature'] = predicted_max
        if predicted_min is not None:
            climate_combined_df.loc[mask_2018, 'Min Temperature'] = predicted_min

print("Completed missing value imputation for 2018 data.")


Completed missing value imputation for 2018 data.


In [96]:

# Before and after missing value counts
missing_before = climate_combined_df[climate_combined_df['Year'] == 2018][
    ['Max Temperature', 'Min Temperature']
].isna().sum()

missing_after = climate_combined_df[climate_combined_df['Year'] == 2018][
    ['Max Temperature', 'Min Temperature']
].isna().sum()

print("Missing values in 2018 data:")
print(f"- Before imputation: Max Temp - {missing_before['Max Temperature']}, Min Temp - {missing_before['Min Temperature']}")
print(f"- After imputation: Max Temp - {missing_after['Max Temperature']}, Min Temp - {missing_after['Min Temperature']}")


Missing values in 2018 data:
- Before imputation: Max Temp - 0, Min Temp - 0
- After imputation: Max Temp - 0, Min Temp - 0


In [97]:

def process_humidity_data(file_path):
    """
    Converts monthly humidity data to daily estimates with realistic variations.
    
    Parameters:
    - file_path: Path to CSV file containing monthly humidity data
    
    Returns:
    - DataFrame with daily humidity estimates
    """
    # Load and prepare data
    humidity_df = pd.read_csv(file_path)
    print(humidity_df.head())
    # Load and prepare data
    print(f"\nColumns in {file_path}: {humidity_df.columns.tolist()}")
    print(humidity_df.head())

    # Month name mapping
    month_name_to_num = {
        'JAN': 1, 'FEB': 2, 'MAR': 3, 'APR': 4,
        'MAY': 5, 'JUN': 6, 'JUL': 7, 'AUG': 8,
        'SEP': 9, 'OCT': 10, 'NOV': 11, 'DEC': 12
    }
    humidity_df['Month_num'] = humidity_df['Month'].map(month_name_to_num)
    
    # Process each monthly record
    daily_humidity_records = []
    
    for _, row in humidity_df.iterrows():
        year = int(row['Year'])
        month = int(row['Month_num'])
        region = row['Region']
        monthly_humidity = float(row['Value'])
        
        # Create daily values with realistic variation
        days_in_month = pd.Timestamp(year, month, 1).days_in_month
        daily_humidity = np.random.normal(
            loc=monthly_humidity,
            scale=1.5,  # Small variation for humidity
            size=days_in_month
        )
        daily_humidity = np.clip(daily_humidity, 0, 100)  # Valid humidity range
        
        # Create daily records
        for day in range(1, days_in_month + 1):
            record_date = datetime.datetime(year, month, day).strftime("%Y-%m-%d")
            daily_humidity_records.append({
                'District': region,  # Will match with district in main DF
                'Date': record_date,
                'Humidity': round(daily_humidity[day - 1], 1)
            })
    
    return pd.DataFrame(daily_humidity_records)

# Process all humidity files


In [98]:
import os
import pandas as pd

# Define district-to-island mapping
PEMBA_DISTRICTS = ["CHAKE-CHAKE", "MKOANI", "WETE", "MICHEWENI"]
UNGUJA_DISTRICTS = ["MAGHARIBI A", "MAGHARIBI B", "KASKAZINI A", "KASKAZINI B", "KATI", "KUSINI", "MJINI"]

# Month map
MONTH_MAP = {
    "JAN": 1, "FEB": 2, "MAR": 3, "APR": 4,
    "MAY": 5, "JUN": 6, "JUL": 7, "AUG": 8,
    "SEP": 9, "OCT": 10, "NOV": 11, "DEC": 12
}

def get_island(region):
    region = region.strip().upper()
    if region in PEMBA_DISTRICTS:
        return "Pemba"
    elif region in UNGUJA_DISTRICTS:
        return "Unguja"
    return "UNKNOWN"

humidity_folder = '../RawDataset/humidity'
all_data = []

print("Processing humidity data files:")
for filename in os.listdir(humidity_folder):
    if filename.endswith('.csv'):
        file_path = os.path.join(humidity_folder, filename)
        print(f"- {filename}")
        df = pd.read_csv(file_path)
        df.columns = [col.strip() for col in df.columns]

        df['Region'] = df['Region'].str.strip().str.upper()
        df['Month'] = df['Month'].str.strip().str.upper()
        df['Month'] = df['Month'].map(MONTH_MAP)
        df['Island'] = df['Region'].apply(get_island)
        df['Humidity'] = df['Value']
        df['Date'] = pd.to_datetime(dict(year=df['Year'], month=df['Month'], day=1))
        df = df[['Date', 'Humidity', 'Region', 'Island']]
        all_data.append(df)

# Combine all files
combined_df = pd.concat(all_data, ignore_index=True)
combined_df.sort_values(by=["Region", "Date"], inplace=True)

# Save raw combined version
combined_df.to_csv("../RawDataset/combined_humidity.csv", index=False)

# Reconstruct a complete (Date, Region) matrix and fill missing
def generate_complete_matrix(regions, df):
    all_dates = pd.date_range(start=df['Date'].min(), end=df['Date'].max(), freq='MS')
    filled_data = []

    global_avg = df['Humidity'].mean()

    for region in regions:
        island = get_island(region)
        region_df = df[df['Region'] == region]
        island_df = df[df['Island'] == island]

        for date in all_dates:
            row = region_df[region_df['Date'] == date]
            if not row.empty:
                filled_data.append(row.iloc[0])
            else:
                # Try same-island other region
                island_candidates = island_df[island_df['Date'] == date]
                if not island_candidates.empty:
                    humidity = island_candidates['Humidity'].mean()
                else:
                    # Fallback: same region average
                    same_region_avg = region_df['Humidity'].mean()
                    if pd.notnull(same_region_avg):
                        humidity = same_region_avg
                    else:
                        # Final fallback: global average
                        humidity = global_avg

                filled_data.append(pd.Series({
                    'Date': date,
                    'Humidity': humidity,
                    'Region': region,
                    'Island': island
                }))
    return pd.DataFrame(filled_data)


# Apply for both islands
pemba_df = generate_complete_matrix(PEMBA_DISTRICTS, combined_df)
unguja_df = generate_complete_matrix(UNGUJA_DISTRICTS, combined_df)

# Combine and save
final_df = pd.concat([pemba_df, unguja_df], ignore_index=True)
final_df.sort_values(by=['Region', 'Date'], inplace=True)
final_df.to_csv("../RawDataset/final_humidity_filled_all_regions.csv", index=False)

print("\nFinal filled humidity data sample:")
print(final_df.head())
print(final_df.shape)

Processing humidity data files:
- humidity_magharibi_a.csv
- pemba_humidity.csv
- humidity_magharibi_b.csv

Final filled humidity data sample:
        Date  Humidity       Region Island
0 2018-01-01      90.0  CHAKE-CHAKE  Pemba
1 2018-02-01      90.0  CHAKE-CHAKE  Pemba
2 2018-03-01      92.0  CHAKE-CHAKE  Pemba
3 2018-04-01      95.0  CHAKE-CHAKE  Pemba
4 2018-05-01      94.0  CHAKE-CHAKE  Pemba
(792, 4)


In [99]:
import pandas as pd

# Load the monthly dataset
monthly_df = pd.read_csv("../RawDataset/final_humidity_filled_all_regions.csv")
monthly_df['Date'] = pd.to_datetime(monthly_df['Date'])

# Ensure correct columns
assert set(['Date', 'Humidity', 'Region', 'Island']).issubset(monthly_df.columns)

# Set date index
monthly_df.set_index('Date', inplace=True)

# Store results
weekly_data = []

for region in monthly_df['Region'].unique():
    region_df = monthly_df[monthly_df['Region'] == region].copy()
    island = region_df['Island'].iloc[0]

    # Resample only humidity
    humidity_weekly = region_df['Humidity'].resample('W').mean()

    # Interpolate missing weekly humidity using linear method
    humidity_weekly = humidity_weekly.interpolate(method='linear', limit_direction='both')

    # Build final weekly DataFrame
    df_weekly = pd.DataFrame({
        'Date': humidity_weekly.index,
        'Humidity': humidity_weekly.values,
        'Region': region,
        'Island': island,
    })

    # Add year and ISO week number
    df_weekly['Year'] = df_weekly['Date'].dt.isocalendar().year
    df_weekly['Week'] = df_weekly['Date'].dt.isocalendar().week

    weekly_data.append(df_weekly)

# Combine all regions' weekly data
final_weekly_df = pd.concat(weekly_data, ignore_index=True)

# Optional: keep only weeks 1–52 (some years have week 53)
final_weekly_df = final_weekly_df[final_weekly_df['Week'] <= 52]

# Sort for clarity
final_weekly_df.sort_values(by=['Region', 'Year', 'Week'], inplace=True)

# Save final output
final_weekly_df.to_csv("../RawDataset/final_weekly_humidity_data.csv", index=False)

print("✅ Final weekly humidity data generated:")
print(final_weekly_df.head(10))


✅ Final weekly humidity data generated:
        Date  Humidity       Region Island  Year  Week
0 2018-01-07     90.00  CHAKE-CHAKE  Pemba  2018     1
1 2018-01-14     90.00  CHAKE-CHAKE  Pemba  2018     2
2 2018-01-21     90.00  CHAKE-CHAKE  Pemba  2018     3
3 2018-01-28     90.00  CHAKE-CHAKE  Pemba  2018     4
4 2018-02-04     90.00  CHAKE-CHAKE  Pemba  2018     5
5 2018-02-11     90.50  CHAKE-CHAKE  Pemba  2018     6
6 2018-02-18     91.00  CHAKE-CHAKE  Pemba  2018     7
7 2018-02-25     91.50  CHAKE-CHAKE  Pemba  2018     8
8 2018-03-04     92.00  CHAKE-CHAKE  Pemba  2018     9
9 2018-03-11     92.75  CHAKE-CHAKE  Pemba  2018    10


In [2]:
import os
import pandas as pd
import numpy as np
import datetime

PEMBA_DISTRICTS = ['WETE', 'MICHEWENI', 'CHAKE-CHAKE', 'MKOANI']
UNGUJA_DISTRICTS = ['MAGHARIBI  B', 'KUSINI', 'MJINI', 'KASKAZINI B', 'MAGHARIBI  A', 'KASKAZINI A', 'KATI']
ALL_DISTRICTS = PEMBA_DISTRICTS + UNGUJA_DISTRICTS

def process_humidity_data(file_path):
    """
    Converts monthly humidity data to weekly averages and fills missing districts.
    """
    humidity_df = pd.read_csv(file_path)
    month_name_to_num = {
        'JAN': 1, 'FEB': 2, 'MAR': 3, 'APR': 4,
        'MAY': 5, 'JUN': 6, 'JUL': 7, 'AUG': 8,
        'SEP': 9, 'OCT': 10, 'NOV': 11, 'DEC': 12
    }
    humidity_df['Month_num'] = humidity_df['Month'].map(month_name_to_num)

    daily_humidity_records = []
    for _, row in humidity_df.iterrows():
        year = int(row['Year'])
        month = int(row['Month_num'])
        region = row['Region'].strip().upper()
        monthly_humidity = float(row['Value'])

        days_in_month = pd.Timestamp(year, month, 1).days_in_month
        daily_humidity = np.random.normal(loc=monthly_humidity, scale=1.5, size=days_in_month)
        daily_humidity = np.clip(daily_humidity, 0, 100)

        for day in range(1, days_in_month + 1):
            date_obj = datetime.date(year, month, day)
            daily_humidity_records.append({
                'District': region,
                'Date': date_obj,
                'Humidity': round(daily_humidity[day - 1], 1)
            })

    daily_df = pd.DataFrame(daily_humidity_records)
    daily_df['Date'] = pd.to_datetime(daily_df['Date'])

    isocal = daily_df['Date'].dt.isocalendar()
    daily_df['Year'] = isocal['year']
    daily_df['Week'] = isocal['week']
    daily_df = daily_df[daily_df['Week'] <= 52]

    weekly_df = (
        daily_df.groupby(['District', 'Year', 'Week'])['Humidity']
        .mean()
        .reset_index()
    )

    return weekly_df


def fill_missing_districts(df):
    """
    Ensures every (Year, Week) has all known districts.
    If a district is missing, copy from a similar district in the same island.
    """
    filled_rows = []
    all_weeks = df[['Year', 'Week']].drop_duplicates()

    for _, row in all_weeks.iterrows():
        year, week = row['Year'], row['Week']
        week_df = df[(df['Year'] == year) & (df['Week'] == week)]
        present_districts = set(week_df['District'])
        missing_districts = set(ALL_DISTRICTS) - present_districts

        for missing in missing_districts:
            island_group = PEMBA_DISTRICTS if missing in PEMBA_DISTRICTS else UNGUJA_DISTRICTS
            # Find source district from same island
            source_row = week_df[week_df['District'].isin(island_group)]
            if not source_row.empty:
                source_data = source_row.iloc[0].copy()
                source_data['District'] = missing
                filled_rows.append(source_data)

    filled_df = pd.DataFrame(filled_rows)
    full_df = pd.concat([df, filled_df], ignore_index=True)
    return full_df.sort_values(by=["District", "Year", "Week"])


# Process all humidity files
humidity_folder = '../RawDataset/humidity'
all_humidity_data = []

print("Processing humidity data files:")
for filename in os.listdir(humidity_folder):
    if filename.endswith('.csv'):
        file_path = os.path.join(humidity_folder, filename)
        print(f"- {filename}")
        humidity_df = process_humidity_data(file_path)
        all_humidity_data.append(humidity_df)

# Combine and fill missing districts
combined_humidity_df = pd.concat(all_humidity_data, ignore_index=True)
combined_humidity_df = fill_missing_districts(combined_humidity_df)

# Save result
combined_humidity_df.to_csv("../RawDataset/combined_weekly_humidity.csv", index=False)

print("\nFinal humidity shape (should have all districts for all weeks):")
print(combined_humidity_df.shape)
print(combined_humidity_df.head())


Processing humidity data files:
- humidity_magharibi_a.csv
- pemba_humidity.csv
- humidity_magharibi_b.csv

Final humidity shape (should have all districts for all weeks):
(3432, 4)
        District  Year  Week   Humidity
312  CHAKE-CHAKE  2018     1  89.871429
313  CHAKE-CHAKE  2018     2  90.200000
314  CHAKE-CHAKE  2018     3  89.942857
315  CHAKE-CHAKE  2018     4  90.642857
316  CHAKE-CHAKE  2018     5  90.000000
